In [1]:
# Importar librerías
import pandas as pd
import os
from src.ingestion import *
from src.config import cols_resultados, cols_balance, cols_cashflow

In [2]:
# Generar lista de tickers del fichero constituents del S&P 500
data_folder = "data"
os.makedirs(data_folder, exist_ok=True) # Crear carpeta si no existe
df_tickers = pd.read_csv(f"{data_folder}/constituents.csv")

# Modificar "BRK.B" a "BRK-B" y "BF.B" a "BF-B" para evitar problemas con yfinance
df_tickers["Symbol"] = df_tickers["Symbol"].replace("BRK.B", "BRK-B")
df_tickers["Symbol"] = df_tickers["Symbol"].replace("BF.B", "BF-B")

tickers_list = df_tickers["Symbol"].tolist()
len(tickers_list)

502

In [3]:
# Seleccionar y renombrar columnas en df_tickers
df_tickers = df_tickers[["Symbol", "GICS Sector", "Date added"]]
df_tickers.rename(columns={
    "Symbol": "Ticker",
    "GICS Sector": "Sector",
    "Date added": "DateAdded"
    }, inplace=True)

# Eliminar espacios en los nombres de los sectores
df_tickers["Sector"] = df_tickers["Sector"].str.replace(" ", "")

df_tickers.head()

,Ticker,Sector,DateAdded
0,MMM,Industrials,1957-03-04
1,AOS,Industrials,2017-07-26
2,ABT,HealthCare,1957-03-04
3,ABBV,HealthCare,2012-12-31
4,ACN,InformationTechnology,2011-07-06


In [4]:
# Extraer precios de los tickers y del índice SPY (demora unos 3 minutos)
df_prices = extraer_precios(tickers_list)
df_index = extraer_precios(['SPY'])
df_prices.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23947 entries, 0 to 23946
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Fecha   23947 non-null  datetime64[ns]
 1   Close   23947 non-null  float64       
 2   Ticker  23947 non-null  object        
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 561.4+ KB


In [5]:
# Calcular los retornos mensuales, varianza del activo y covarianza con el mercado para cada ticker
df_retornos = calcular_retornos(df_prices, df_index)

# Se renombran las columnas
df_retornos = df_retornos.rename(columns={
    'Fecha' : 'Date',
    'Retorno_Mensual' : 'Monthly_Return',
    'Varianza_Activo': 'Monthly_Variance',
    'Covarianza_Mercado' : 'Market_Covariance'
})
# Quitar zonas horarias de Date y convertir a object con fecha pura
df_retornos['Date'] = df_retornos['Date'].dt.tz_localize(None).dt.date

df_retornos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19432 entries, 0 to 19431
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Date               19432 non-null  object 
 1   Ticker             19432 non-null  object 
 2   Monthly_Return     19430 non-null  float64
 3   Monthly_Variance   19432 non-null  float64
 4   Market_Covariance  19432 non-null  float64
dtypes: float64(3), object(2)
memory usage: 759.2+ KB


In [6]:
# Extraer datos financieros de yfinance (demora 10 minutos)
df_financials = extraer_financials(tickers_list)
df_financials.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2376 entries, 0 to 2375
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Fecha                      2376 non-null   object 
 1   Total Revenue              2007 non-null   float64
 2   Gross Profit               1797 non-null   float64
 3   Operating Income           1823 non-null   float64
 4   Net Income                 1999 non-null   float64
 5   EBITDA                     1815 non-null   float64
 6   Basic Average Shares       2000 non-null   float64
 7   Cash And Cash Equivalents  2000 non-null   float64
 8   Current Debt               1592 non-null   float64
 9   Long Term Debt             1894 non-null   float64
 10  Total Debt                 1988 non-null   float64
 11  Stockholders Equity        2000 non-null   float64
 12  Total Assets               2004 non-null   float64
 13  Current Assets             1820 non-null   float

In [7]:
# Se renombran las columnas Fecha a Date
df_financials = df_financials.rename(columns={'Fecha': 'Date'})
df_prices = df_prices.rename(columns={'Fecha': 'Date'})

# Eliminar cualquier columna duplicada (para evitar errores al reejecutar la celda)
df_financials = df_financials.loc[:, ~df_financials.columns.duplicated()]
df_prices = df_prices.loc[:, ~df_prices.columns.duplicated()]

# Se convierten a datetime nativo para poder hacer cálculos
df_financials['Date'] = pd.to_datetime(df_financials['Date']).dt.tz_localize(None)
df_prices['Date'] = pd.to_datetime(df_prices['Date']).dt.tz_localize(None)

# Cálculo del primer día del mes siguiente
df_financials['Date'] = (df_financials['Date'] + pd.offsets.MonthEnd(0)) + pd.Timedelta(days=1)

# Se convierten ambos DataFrames a fecha pura sin hora
df_financials['Date'] = df_financials['Date'].dt.date
df_prices['Date'] = df_prices['Date'].dt.date

# Asegurar que los Tickers no tengan espacios en blanco invisibles
df_prices['Ticker'] = df_prices['Ticker'].astype(str).str.strip()
df_financials['Ticker'] = df_financials['Ticker'].astype(str).str.strip()

# Unir datos financieros con precios mensuales
df_merged = pd.merge(
    df_prices, 
    df_financials, 
    on=['Date', 'Ticker'],
    how='left'
)

# Ordenar cronológicamente para el Forward Fill
df_merged = df_merged.sort_values(by=['Ticker', 'Date'])

# Aplicar Forward Fill a las columnas financieras
cols_financieras = cols_resultados + cols_balance + cols_cashflow

df_merged[cols_financieras] = df_merged.groupby('Ticker')[cols_financieras].ffill()

df_merged.info()


<class 'pandas.core.frame.DataFrame'>
Index: 23947 entries, 432 to 23946
Data columns (total 22 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Date                       23947 non-null  object 
 1   Close                      23947 non-null  float64
 2   Ticker                     23947 non-null  object 
 3   Total Revenue              19861 non-null  float64
 4   Gross Profit               17803 non-null  float64
 5   Operating Income           18064 non-null  float64
 6   Net Income                 19783 non-null  float64
 7   EBITDA                     17986 non-null  float64
 8   Basic Average Shares       19810 non-null  float64
 9   Cash And Cash Equivalents  19822 non-null  float64
 10  Current Debt               16520 non-null  float64
 11  Long Term Debt             18863 non-null  float64
 12  Total Debt                 19798 non-null  float64
 13  Stockholders Equity        19822 non-null  float6

In [8]:
# Eliminar las filas anteriores al primer reporte financiero disponible
columna_critica = 'EBITDA' # es necesaria para los ratios
df_merged_clean = df_merged.dropna(subset=[columna_critica])

# Resetear el índice para que quede estético de 0 a N
df_merged_clean = df_merged_clean.reset_index(drop=True)

# Unir los retornos calculados al dataframe limpio final
df_merged_clean = pd.merge(df_merged_clean, df_retornos, on=['Ticker', 'Date'], how='left')

df_merged_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17986 entries, 0 to 17985
Data columns (total 25 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Date                       17986 non-null  object 
 1   Close                      17986 non-null  float64
 2   Ticker                     17986 non-null  object 
 3   Total Revenue              17986 non-null  float64
 4   Gross Profit               17725 non-null  float64
 5   Operating Income           17986 non-null  float64
 6   Net Income                 17947 non-null  float64
 7   EBITDA                     17986 non-null  float64
 8   Basic Average Shares       17935 non-null  float64
 9   Cash And Cash Equivalents  17947 non-null  float64
 10  Current Debt               14995 non-null  float64
 11  Long Term Debt             17066 non-null  float64
 12  Total Debt                 17923 non-null  float64
 13  Stockholders Equity        17947 non-null  flo

In [9]:
df_with_metrics = calcular_metricas(df_merged_clean)

# Luego de calcular los features, se eliminan las columnas financieras originales
df_with_metrics = df_with_metrics.drop(columns=cols_financieras)
df_with_metrics.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17986 entries, 0 to 17985
Data columns (total 28 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Date                17986 non-null  object 
 1   Close               17986 non-null  float64
 2   Ticker              17986 non-null  object 
 3   Monthly_Return      17514 non-null  float64
 4   Monthly_Variance    17514 non-null  float64
 5   Market_Covariance   17514 non-null  float64
 6   MarketCap           17935 non-null  float64
 7   EnterpriseValue     17935 non-null  float64
 8   PE_Trailing         17896 non-null  float64
 9   EnterpriseToEbitda  17935 non-null  float64
 10  PriceToBook         17896 non-null  float64
 11  operatingMargins    17986 non-null  float64
 12  profitMargins       17947 non-null  float64
 13  returnOnEquity      17908 non-null  float64
 14  ReturnOnAssets      17947 non-null  float64
 15  debtToEquity        17947 non-null  float64
 16  curr

In [10]:
# Unir df_with_metrics con df_tickers
df_final = pd.merge(df_with_metrics, df_tickers, on="Ticker", how="left")
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17986 entries, 0 to 17985
Data columns (total 30 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Date                17986 non-null  object 
 1   Close               17986 non-null  float64
 2   Ticker              17986 non-null  object 
 3   Monthly_Return      17514 non-null  float64
 4   Monthly_Variance    17514 non-null  float64
 5   Market_Covariance   17514 non-null  float64
 6   MarketCap           17935 non-null  float64
 7   EnterpriseValue     17935 non-null  float64
 8   PE_Trailing         17896 non-null  float64
 9   EnterpriseToEbitda  17935 non-null  float64
 10  PriceToBook         17896 non-null  float64
 11  operatingMargins    17986 non-null  float64
 12  profitMargins       17947 non-null  float64
 13  returnOnEquity      17908 non-null  float64
 14  ReturnOnAssets      17947 non-null  float64
 15  debtToEquity        17947 non-null  float64
 16  curr

In [11]:
# Extraer datos macro (devuelve dataframe vacio si no se configuro la clave API de FRED)
indicadores = [
    'FEDFUNDS',     # Tasa de Fondos Federales (Tasa de interés de política monetaria de la Fed)
    'CPIAUCSL',     # Índice de Precios al Consumidor (IPC / Inflación general en EE.UU.)
    'UNRATE',       # Tasa de Desempleo oficial de EE.UU.
    'T10Y2Y',       # Diferencial de la curva de rendimientos (10 años menos 2 años, indicador de recesión)
    'BAMLH0A0HYM2', # Diferencial de bonos "High Yield" vs Tesoro (Termómetro de estrés de liquidez y riesgo de default corporativo)
    'M2SL'          # Oferta Monetaria M2 (Liquidez total en la economía, clave para entender la expansión de activos financieros)
]

df_datos_macro = extraer_datos_macro(indicadores)
df_datos_macro = df_datos_macro.sort_values(by='Fecha')
df_datos_macro.head()

,Fecha,FEDFUNDS,CPIAUCSL,UNRATE,T10Y2Y,BAMLH0A0HYM2,M2SL
0,2022-06-30,1.21,294.957,3.6,0.06,NaN,21651.2
1,2022-07-31,1.68,294.913,3.5,-0.22,NaN,21652.8
2,2022-08-31,2.33,295.097,3.6,-0.30,NaN,21638.2
3,2022-09-30,2.56,296.349,3.5,-0.39,NaN,21536.5
4,2022-10-31,3.08,298.007,3.6,-0.41,NaN,21461.5


In [12]:
# Se renombran columnas
df_datos_macro = df_datos_macro.rename(columns={
    'FEDFUNDS': 'FedFundsRate',
    'CPIAUCSL' : "CPI",
    'UNRATE': 'UnemploymentRate',
    'T10Y2Y' : '10Y2YSpread',
    'BAMLH0A0HYM2': 'HighYieldSpread',
    'M2SL' : "M2"
})

# Se convierten las series con crecimiento infinito a variaciones porcentuales trimestrales (QoQ) y anuales (YoY)
variables_a_convertir = ['CPI', 'M2']
df_datos_macro = calcular_growth_features(df_datos_macro, variables_a_convertir)

# Se eliminan las columnas originales
df_datos_macro = df_datos_macro.drop(variables_a_convertir, axis=1)

# Asegurar que la columna Fecha sea tipo datetime
df_datos_macro = df_datos_macro.rename(columns = {'Fecha' : 'Date'})
df_datos_macro['Date'] = pd.to_datetime(df_datos_macro['Date'])

# 2. Desplazar la fecha al 1er día del mes, saltando un mes de publicación para evitar fuga de datos
# Ej: 2022-06-30 -> 2022-08-01
df_datos_macro['Date'] = df_datos_macro['Date'] + pd.offsets.MonthBegin(2)

# Se convierte Date a tipo Object
df_datos_macro['Date'] = df_datos_macro['Date'].dt.date

In [13]:
# Se une con los datos de df_final
df_raw = pd.merge(df_final, df_datos_macro, on="Date", how= "left")
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17986 entries, 0 to 17985
Data columns (total 38 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Date                17986 non-null  object 
 1   Close               17986 non-null  float64
 2   Ticker              17986 non-null  object 
 3   Monthly_Return      17514 non-null  float64
 4   Monthly_Variance    17514 non-null  float64
 5   Market_Covariance   17514 non-null  float64
 6   MarketCap           17935 non-null  float64
 7   EnterpriseValue     17935 non-null  float64
 8   PE_Trailing         17896 non-null  float64
 9   EnterpriseToEbitda  17935 non-null  float64
 10  PriceToBook         17896 non-null  float64
 11  operatingMargins    17986 non-null  float64
 12  profitMargins       17947 non-null  float64
 13  returnOnEquity      17908 non-null  float64
 14  ReturnOnAssets      17947 non-null  float64
 15  debtToEquity        17947 non-null  float64
 16  curr

In [14]:
# Guardar datos extraidos en fichero raw_data
df_raw.to_parquet(f"{data_folder}/raw_data.parquet")